# 🌟 GraphRAG++ Fine-Tuning (4-bit, A100)
## `Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled`

| Setting | Value |
|---|---|
| GPU | A100 80 GB |
| Precision | **4-bit QLoRA** (model ~14 GB, 65 GB headroom) |
| LoRA | r=32, RSLoRA |
| Seq length | 4096 |
| Eff. batch | 16 (2 × 8) |

**Why 4-bit?** Full bf16 (27B × 2 bytes = 54 GB) leaves only ~5 GB for activations → OOM.  
4-bit (27B × 0.5 bytes = ~14 GB) gives 65 GB headroom — plenty for batch=2 + optimizer states.

In [ ]:
# ── Cell 1: env flags (MUST run before any import) ───────────────────────────
import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'       # skip HF stats ping
os.environ['PYTORCH_CUDA_ALLOC_CONF']    = 'expandable_segments:True'

import torch
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU   : {torch.cuda.get_device_name(0)}')
print(f'VRAM  : {vram:.0f} GB')
print(f'Torch : {torch.__version__}  CUDA: {torch.version.cuda}')

In [ ]:
# ── Cell 2: install ──────────────────────────────────────────────────────────
!pip install unsloth -q --upgrade
!pip install --no-deps trl peft accelerate bitsandbytes -q
!pip install datasets huggingface_hub -q
import unsloth; print(f'Unsloth {unsloth.__version__}')

In [ ]:
# ── Cell 3: HuggingFace login ────────────────────────────────────────────────
from huggingface_hub import login
HF_TOKEN = ''   # ← paste your token (huggingface.co/settings/tokens)
if HF_TOKEN:
    login(token=HF_TOKEN)
    print('Logged in')
else:
    print('No token — skipping login')

In [ ]:
# ── Cell 4: load model in 4-bit ──────────────────────────────────────────────
# 4-bit: model ≈14 GB  →  ~65 GB free for activations + optimizer states
from unsloth import FastLanguageModel
import torch, gc

MODEL_ID    = 'Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled'
MAX_SEQ_LEN = 4096

print(f'Loading {MODEL_ID} in 4-bit ...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LEN,
    dtype          = None,      # auto (bf16 for compute, 4-bit storage)
    load_in_4bit   = True,      # ← 4-bit NF4 quantization
    token          = HF_TOKEN or None,
)

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Loaded  |  VRAM used: {used:.1f} GB / {total:.0f} GB  ({used/total*100:.0f}%)')
print(f'Free for training: {total - used:.0f} GB')

In [ ]:
# ── Cell 4b: extract plain text tokenizer ───────────────────────────────────
# Qwen3.5 is multimodal — Unsloth returns Qwen2VLProcessor.
# __call__(images=None, text=None) → tokenizer("str") routes to images → crash.
# Fix: extract the embedded AutoTokenizer for all SFT steps.

if hasattr(tokenizer, 'tokenizer'):
    text_tok = tokenizer.tokenizer
    if text_tok.pad_token is None:
        text_tok.pad_token = text_tok.eos_token
    print(f'Multimodal processor → extracted text tokenizer: {type(text_tok).__name__}')
else:
    text_tok = tokenizer
    print(f'Plain tokenizer: {type(text_tok).__name__}')

EOS = text_tok.eos_token
print(f'EOS token: {repr(EOS)}')

In [ ]:
# ── Cell 5: LoRA adapters ────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r                          = 32,
    target_modules             = ['q_proj','k_proj','v_proj','o_proj',
                                   'gate_proj','up_proj','down_proj'],
    lora_alpha                 = 64,
    lora_dropout               = 0.05,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
    use_rslora                 = True,
    loftq_config               = None,
)
tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f'LoRA r=32, alpha=64, RSLoRA')
print(f'Trainable: {tr/1e6:.1f}M  ({tr/tot*100:.2f}%)')

In [ ]:
# ── Cell 6: dataset ──────────────────────────────────────────────────────────
EXAMPLES = [
    {
        'instruction': 'Extract entity-relation triples from this text as JSON.',
        'input': 'The Transformer was introduced by Vaswani et al. in 2017. BERT, developed by Google, uses bidirectional transformers pre-trained on Wikipedia and BookCorpus.',
        'output': '<think>\nTransformer introduced_by Vaswani 2017. BERT developed_by Google, uses bidirectional transformers, trained_on Wikipedia/BookCorpus, based_on Transformer.\n</think>\n[{"subject":"Transformer","predicate":"introduced_by","object":"Vaswani et al.","confidence":0.99},{"subject":"Transformer","predicate":"published_in","object":"2017","confidence":0.99},{"subject":"BERT","predicate":"developed_by","object":"Google","confidence":0.97},{"subject":"BERT","predicate":"uses","object":"bidirectional transformers","confidence":0.97},{"subject":"BERT","predicate":"trained_on","object":"Wikipedia","confidence":0.96},{"subject":"BERT","predicate":"based_on","object":"Transformer","confidence":0.94}]'
    },
    {
        'instruction': 'Extract entity-relation triples from this codebase description as JSON.',
        'input': 'vLLM uses PagedAttention for KV cache management. It integrates with Ray for distributed inference and was developed at UC Berkeley.',
        'output': '<think>\nvLLM uses PagedAttention, integrates_with Ray, developed_at UC Berkeley.\n</think>\n[{"subject":"vLLM","predicate":"uses","object":"PagedAttention","confidence":0.98},{"subject":"vLLM","predicate":"integrates_with","object":"Ray","confidence":0.95},{"subject":"vLLM","predicate":"developed_at","object":"UC Berkeley","confidence":0.95}]'
    },
    {
        'instruction': 'Extract entity-relation triples from this ML paper abstract as JSON.',
        'input': 'LoRA freezes pre-trained weights and injects trainable rank decomposition matrices into each Transformer layer, reducing trainable parameters by 10000x vs Adam fine-tuning.',
        'output': '<think>\nLoRA freezes weights, injects matrices, applied_to Transformer, reduces params.\n</think>\n[{"subject":"LoRA","predicate":"freezes","object":"pre-trained weights","confidence":0.98},{"subject":"LoRA","predicate":"injects","object":"rank decomposition matrices","confidence":0.98},{"subject":"LoRA","predicate":"applied_to","object":"Transformer layers","confidence":0.97},{"subject":"LoRA","predicate":"reduces","object":"trainable parameters","confidence":0.97}]'
    },
    {
        'instruction': 'Classify this query intent. Output JSON: {"intent": "factual|multi-hop|aggregative|comparative", "entities": [...], "confidence": 0.0}',
        'input': 'How does LoRA enable efficient fine-tuning, and which frameworks implement it?',
        'output': '<think>\nRelational + aggregation over frameworks = multi-hop.\n</think>\n{"intent":"multi-hop","entities":["LoRA","efficient fine-tuning"],"confidence":0.93}'
    },
    {
        'instruction': 'Classify this query intent. Output JSON: {"intent": "factual|multi-hop|aggregative|comparative", "entities": [...], "confidence": 0.0}',
        'input': 'What is PagedAttention?',
        'output': '<think>\nSingle entity definitional lookup. factual.\n</think>\n{"intent":"factual","entities":["PagedAttention"],"confidence":0.97}'
    },
    {
        'instruction': 'Classify this query intent. Output JSON: {"intent": "factual|multi-hop|aggregative|comparative", "entities": [...], "confidence": 0.0}',
        'input': 'Compare VRAM and throughput of LoRA vs QLoRA vs full fine-tuning.',
        'output': '<think>\nMultiple entities, multiple comparison dimensions. comparative.\n</think>\n{"intent":"comparative","entities":["LoRA","QLoRA","full fine-tuning"],"confidence":0.91}'
    },
    {
        'instruction': 'Given these knowledge graph paths, answer the question.',
        'input': 'Question: What enables efficient LLM fine-tuning?\nPath 1 (0.97): LoRA --[enables]--> PEFT --[reduces]--> Training Cost\nPath 2 (0.96): QLoRA --[extends]--> LoRA\nPath 3 (0.95): Unsloth --[optimizes]--> LoRA --[reduces]--> VRAM',
        'output': '<think>\n3 paths on LoRA-family: mechanism, variant, tooling.\n</think>\n1. **LoRA** enables PEFT, reducing cost [Path 1, 0.97]\n2. **QLoRA** adds 4-bit quant [Path 2, 0.96]\n3. **Unsloth** optimizes VRAM/speed [Path 3, 0.95]'
    },
    {
        'instruction': 'Given these knowledge graph paths, answer the question.',
        'input': 'Question: How is PyTorch connected to production LLM infrastructure?\nPath 1 (0.97): PyTorch --[powers]--> Transformer models --[served_by]--> vLLM\nPath 2 (0.96): PyTorch --[foundation_of]--> HF Transformers --[provides]--> Model Hub',
        'output': '<think>\n2 paths from PyTorch to production.\n</think>\n1. PyTorch → Transformer models → vLLM [Path 1, 0.97]\n2. PyTorch → HF Transformers → Model Hub → vLLM [Path 2, 0.96]'
    },
    {
        'instruction': 'Given knowledge graph evidence, answer with counterfactual verification.',
        'input': 'Question: What is the relationship between BERT and the Transformer?\nEvidence: BERT based_on Transformer (0.99), uses bidirectional self-attention (0.97), trained_with MLM (0.98)',
        'output': '<think>\nBERT based_on Transformer (0.99). Extends with bidirectionality + MLM. Confirmed valid.\n</think>\nBERT is based on the Transformer encoder [0.99], with bidirectional self-attention [0.97] and MLM pre-training [0.98]. *Counterfactual: confirmed valid.*'
    },
]
print(f'{len(EXAMPLES)} training examples')

In [ ]:
# ── Cell 7: format dataset ───────────────────────────────────────────────────
from datasets import Dataset

ALPACA = (
    "Below is an instruction that describes a task, paired with an input "
    "that provides further context. Write a response that appropriately "
    "completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n{output}"
)

def fmt(ex):
    return {'text': ALPACA.format(**ex) + EOS}

raw      = Dataset.from_list(EXAMPLES)
train_ds = raw.map(fmt, batched=False, remove_columns=raw.column_names)

# Must use text_tok (NOT tokenizer) — tokenizer is Qwen2VLProcessor
lengths = [len(text_tok(ex['text'])['input_ids']) for ex in train_ds]
print(f'Examples : {len(train_ds)}')
print(f'Avg tokens: {sum(lengths)//len(lengths)} | Max: {max(lengths)}')

In [ ]:
# ── Cell 8: train ────────────────────────────────────────────────────────────
import gc, torch
gc.collect(); torch.cuda.empty_cache()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f'VRAM free before training: {free:.1f} GB')

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import train_on_responses_only

trainer = SFTTrainer(
    model              = model,
    tokenizer          = text_tok,       # plain text tokenizer, NOT VL processor
    train_dataset      = train_ds,
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,    # 4096
    dataset_num_proc   = 2,
    packing            = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,   # eff. batch = 16
        warmup_steps                = 5,
        num_train_epochs            = 5,
        learning_rate               = 2e-4,
        fp16                        = False,
        bf16                        = True,
        logging_steps               = 1,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'cosine',
        seed                        = 42,
        output_dir                  = 'graphrag_outputs',
        report_to                   = 'none',
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = '### Instruction:\n',
    response_part    = '### Response:\n',
)

print('Training: 4-bit QLoRA | r=32 RSLoRA | eff-batch=16 | 5 epochs | adamw_8bit')
stats = trainer.train()
print(f'\nDone!  Loss: {stats.training_loss:.4f}  |  {stats.metrics["train_runtime"]/60:.1f} min')

In [ ]:
# ── Cell 9: inference test ───────────────────────────────────────────────────
FastLanguageModel.for_inference(model)

for instruction, inp in [
    ('Extract entity-relation triples as JSON.',
     'GPT-4 from OpenAI uses RLHF. LangChain integrates GPT-4 for RAG pipelines.'),
    ('Classify this query intent. Output JSON.',
     'Which inference frameworks support OpenAI-compatible APIs?'),
]:
    prompt = ALPACA.format(instruction=instruction, input=inp, output='')
    inputs = text_tok([prompt], return_tensors='pt').to('cuda')
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens = 512,
            temperature    = 0.3,
            do_sample      = True,
            pad_token_id   = text_tok.eos_token_id,
        )
    resp = text_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'Q: {instruction}\nA: {resp}\n{"─"*60}')

In [ ]:
# ── Cell 10: save ────────────────────────────────────────────────────────────
OUTPUT = 'graphrag-plus-plus-qwen27b'

# LoRA adapter weights (small — just the delta)
model.save_pretrained(f'{OUTPUT}-lora')
tokenizer.save_pretrained(f'{OUTPUT}-lora')
print(f'Saved: {OUTPUT}-lora/')

# Push to HuggingFace Hub
if HF_TOKEN:
    model.push_to_hub(OUTPUT, token=HF_TOKEN)
    tokenizer.push_to_hub(OUTPUT, token=HF_TOKEN)
    print(f'Pushed: huggingface.co/{OUTPUT}')

# GGUF for Ollama / llama.cpp
print('Exporting GGUF Q4_K_M (~15 GB) ...')
model.save_pretrained_gguf(f'{OUTPUT}-q4', tokenizer, quantization_method='q4_k_m')
print(f'Done! Use with: ollama create graphrag -f {OUTPUT}-q4/Modelfile')